# 📊 주간 트렌드 리포트
**셀 순서대로 실행.** 1번(API 키), 2번(DB 연결) 두 셀만 수정하면 됩니다.

In [ ]:
# ══════════════════════════════════════════════
# CELL 1 ✏️  API 키만 입력
# ══════════════════════════════════════════════

CLAUDE_API_KEY      = "여기에_Claude_API_키"
NAVER_CLIENT_ID     = "여기에_네이버_Client_ID"
NAVER_CLIENT_SECRET = "여기에_네이버_Client_Secret"

TOP_N = 5   # 분석할 상품 수

# 날짜는 자동 계산 (실행 기준 최근 7일)
from datetime import datetime, timedelta
END_DATE   = datetime.now().strftime("%Y-%m-%d")
START_DATE = (datetime.now() - timedelta(days=6)).strftime("%Y-%m-%d")

print("분석 기간:", START_DATE, "~", END_DATE)

In [ ]:
# ══════════════════════════════════════════════
# CELL 2 ✏️  DB 연결 & 상품 데이터 자동 조회
# ══════════════════════════════════════════════
# 비밀번호만 입력하세요. 나머지 수정 불필요.

import prestodb
import pandas as pd

conn = prestodb.dbapi.connect(
    host        = 'kakaoent-presto-adhoc.kakaoent.io',
    port        = 8443,
    user        = 'journi-y222',
    catalog     = 'hadoop_kent',
    schema      = 'ods_commerce_production',
    http_scheme = 'https',
    auth        = prestodb.auth.BasicAuthentication("journi-y222", "여기에_비밀번호"),
)

SQL = """
SELECT
    product_name,
    artist_name,
    COUNT(DISTINCT user_id)                          AS buyer_cnt,
    COUNT(DISTINCT order_id)                         AS order_cnt,
    SUM(IF(amt_rk = 1, COALESCE(krw_amount, 0), 0)) AS gmv
FROM hadoop_kent.journi_y222.mart_daily_order
WHERE completed_dt >= DATE_FORMAT(DATE_ADD('day', -6, CURRENT_DATE), '%Y-%m-%d')
  AND product_name IS NOT NULL
  AND product_name <> ''
GROUP BY product_name, artist_name
ORDER BY buyer_cnt DESC
LIMIT {top_n}
""".format(top_n=TOP_N)

PRODUCTS = pd.read_sql(SQL, conn).to_dict('records')

# 숫자 타입 통일
for p in PRODUCTS:
    p["buyer_cnt"] = int(p.get("buyer_cnt") or 0)
    p["order_cnt"] = int(p.get("order_cnt") or 0)
    p["gmv"]       = int(float(p.get("gmv")  or 0))

print("상품 {}개 로드 완료:".format(len(PRODUCTS)))
for i, p in enumerate(PRODUCTS, 1):
    print("  {}. {} / {} — 구매자 {:,}명".format(
        i, p["artist_name"], p["product_name"], p["buyer_cnt"]))

In [ ]:
# ══════════════════════════════════════════════
# CELL 3  함수 정의 (수정 불필요)
# ══════════════════════════════════════════════

import requests, re

# ── 상품명에서 핵심 키워드 추출 (LIKE 검색용) ────────────────────────
# "청춘 앨범 A ver." → "청춘 앨범"  /  "굿즈 키링 세트" → "굿즈 키링"
def core_keyword(product_name):
    s = re.sub(r'\s+[A-Za-z]\s*(ver|type|edition)\.?', '', product_name, flags=re.IGNORECASE)
    s = re.sub(r'\s+(ver|version|type|edition)\.?\s*[A-Za-z]?', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[\(\)\[\]\{\}]', '', s).strip()
    words = s.split()
    return ' '.join(words[:3]) if len(words) > 3 else s  # 최대 3단어

# ── DataLab: 상품 + 아티스트 동시 조회 (이번주 vs 전주) ─────────────
def get_dual_trend(product_name, artist_name, start, end):
    prev_start = (datetime.strptime(start, "%Y-%m-%d") - timedelta(days=7)).strftime("%Y-%m-%d")
    prev_end   = (datetime.strptime(end,   "%Y-%m-%d") - timedelta(days=7)).strftime("%Y-%m-%d")
    kw_product = core_keyword(product_name)
    try:
        resp = requests.post(
            "https://openapi.naver.com/v1/datalab/search",
            headers={"X-Naver-Client-Id": NAVER_CLIENT_ID,
                     "X-Naver-Client-Secret": NAVER_CLIENT_SECRET,
                     "Content-Type": "application/json"},
            json={
                "startDate": prev_start, "endDate": end, "timeUnit": "date",
                "keywordGroups": [
                    {"groupName": "상품",    "keywords": [kw_product]},   # 상품 핵심어
                    {"groupName": "아티스트", "keywords": [artist_name]}  # 아티스트 전체
                ]
            },
            timeout=10
        )
        results = resp.json().get("results", [])
        def parse(r):
            data = r.get("data", [])
            this_w = [d["ratio"] for d in data if start  <= d["period"] <= end]
            prev_w = [d["ratio"] for d in data if prev_start <= d["period"] <= prev_end]
            this_avg = round(sum(this_w)/len(this_w), 1) if this_w else 0
            prev_avg = round(sum(prev_w)/len(prev_w), 1) if prev_w else 0
            chg = round((this_avg-prev_avg)/prev_avg*100, 1) if prev_avg else 0
            return {"this_avg": this_avg, "prev_avg": prev_avg, "change_pct": chg,
                    "daily": [{"date": d["period"], "ratio": d["ratio"]} for d in data if start <= d["period"] <= end]}
        product_trend  = parse(results[0]) if len(results) > 0 else None
        artist_trend   = parse(results[1]) if len(results) > 1 else None
        return {"product": product_trend, "artist": artist_trend, "kw_product": kw_product}
    except Exception as e:
        print("  DataLab 오류:", e)
        return {"product": None, "artist": None, "kw_product": core_keyword(product_name)}

# ── 뉴스 검색 (상품명 / 아티스트명 각각) ────────────────────────────
def get_news(keyword, n=5):
    TAG = re.compile(r"<[^>]+>")
    def clean(t): return TAG.sub("", t).replace("&quot;", '"').replace("&amp;", "&").strip()
    def parse_date(s):
        try: return datetime.strptime(s[:16], "%a, %d %b %Y").strftime("%Y-%m-%d")
        except: return ""
    try:
        resp = requests.get(
            "https://openapi.naver.com/v1/search/news.json",
            headers={"X-Naver-Client-Id": NAVER_CLIENT_ID, "X-Naver-Client-Secret": NAVER_CLIENT_SECRET},
            params={"query": keyword, "display": 20, "sort": "date"},
            timeout=10
        )
        items = resp.json().get("items", [])
        # 기간 필터 → 없으면 최근 N건
        filtered = []
        for item in items:
            pub = parse_date(item.get("pubDate", ""))
            if pub and START_DATE <= pub <= END_DATE:
                filtered.append({"title": clean(item.get("title","")),
                                 "link":  item.get("originallink") or item.get("link",""),
                                 "desc":  clean(item.get("description","")), "date": pub})
            if len(filtered) >= n: break
        if not filtered:
            for item in items[:n]:
                filtered.append({"title": clean(item.get("title","")),
                                 "link":  item.get("originallink") or item.get("link",""),
                                 "desc":  clean(item.get("description","")),
                                 "date":  parse_date(item.get("pubDate",""))})
        return filtered
    except Exception as e:
        print("  뉴스 오류 ({}):".format(keyword), e)
        return []

# ── 카페글 검색 ──────────────────────────────────────────────────────
def get_cafe(keyword, n=5):
    TAG = re.compile(r"<[^>]+>")
    def clean(t): return TAG.sub("", t).replace("&quot;", '"').replace("&amp;", "&").strip()
    def parse_date(s):
        try: return datetime.strptime(s[:16], "%a, %d %b %Y").strftime("%Y-%m-%d")
        except: return s[:8] if len(s) >= 8 else ""
    try:
        resp = requests.get(
            "https://openapi.naver.com/v1/search/cafearticle.json",
            headers={"X-Naver-Client-Id": NAVER_CLIENT_ID, "X-Naver-Client-Secret": NAVER_CLIENT_SECRET},
            params={"query": keyword, "display": 20, "sort": "date"},
            timeout=10
        )
        items = resp.json().get("items", [])
        result = []
        for item in items[:n]:
            result.append({"title":    clean(item.get("title","")),
                           "link":     item.get("link",""),
                           "cafe":     item.get("cafename",""),
                           "date":     parse_date(item.get("postdate","") or item.get("pubDate",""))})
        return result
    except Exception as e:
        print("  카페 오류 ({}):".format(keyword), e)
        return []

# ── Claude AI 해석 ────────────────────────────────────────────────────
def analyze(product, trend, news_product, news_artist, cafe_product, cafe_artist):
    conv = round(product["buyer_cnt"]/product.get("visitor_cnt",product["buyer_cnt"])*100,1) if product.get("visitor_cnt",0)>0 else "-"

    def trend_txt(t, label):
        if not t: return label + ": 데이터 없음"
        d = "상승" if t["change_pct"]>5 else "하락" if t["change_pct"]<-5 else "보합"
        return "{}: 이번주 {} / 전주 {} ({} {}%)".format(label, t["this_avg"], t["prev_avg"], d, abs(t["change_pct"]))

    def items_txt(items, keys=("title","desc","date")):
        if not items: return "없음"
        return "\n".join(["- [{}] {}: {}".format(i.get(keys[2],""), i.get(keys[0],""), i.get(keys[1] if len(keys)>1 else "title","")) for i in items])

    prompt = (
        "주간 실적 리포트 분석을 부탁드립니다.\n\n"
        "[상품] {} / [아티스트] {} / [기간] {} ~ {}\n"
        "구매자: {:,}명 / 주문: {:,}건 / 거래액: {:,}원\n\n"
        "[네이버 검색 트렌드]\n{}\n{}\n\n"
        "[상품 관련 뉴스 {}건]\n{}\n\n"
        "[아티스트 관련 뉴스 {}건]\n{}\n\n"
        "[상품 관련 팬카페글 {}건]\n{}\n\n"
        "[아티스트 팬카페 반응 {}건]\n{}\n\n"
        "아래 3가지를 간결하게 (전체 250자 이내, **bold** 유지):\n"
        "1. **외부 트렌드 요약** — 이번 주 상품/아티스트 외부 반응 흐름\n"
        "2. **내외부 연관성** — 내부 실적과 외부 버즈의 관계\n"
        "3. **다음 주 전망** — 한 줄로"
    ).format(
        product["product_name"], product["artist_name"], START_DATE, END_DATE,
        product["buyer_cnt"], product["order_cnt"], product["gmv"],
        trend_txt(trend.get("product"), "상품({})".format(trend.get("kw_product",""))),
        trend_txt(trend.get("artist"),  "아티스트({})".format(product["artist_name"])),
        len(news_product),   items_txt(news_product),
        len(news_artist),    items_txt(news_artist),
        len(cafe_product),   items_txt(cafe_product,   ("title","cafe","date")),
        len(cafe_artist),    items_txt(cafe_artist,    ("title","cafe","date")),
    )
    try:
        resp = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={"x-api-key": CLAUDE_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"},
            json={"model": "claude-sonnet-4-5", "max_tokens": 700,
                  "messages": [{"role": "user", "content": prompt}]},
            timeout=30
        )
        return resp.json()["content"][0]["text"]
    except Exception as e:
        print("  Claude 오류:", e)
        return "AI 분석 실패"

print("함수 정의 완료")

In [ ]:
# ══════════════════════════════════════════════
# CELL 4  실행
# ══════════════════════════════════════════════

results = []
for i, product in enumerate(PRODUCTS, 1):
    pname  = product["product_name"]
    aname  = product["artist_name"]
    kw_prd = core_keyword(pname)
    kw_art = aname

    print("\n[{}/{}] {} / {}".format(i, len(PRODUCTS), aname, pname))
    print("  상품 핵심 키워드:", kw_prd)

    # DataLab: 상품 + 아티스트 동시
    print("  DataLab 조회 중...")
    trend = get_dual_trend(pname, aname, START_DATE, END_DATE)
    if trend["product"]: print("  상품 트렌드: 이번주", trend["product"]["this_avg"], "/ 전주", trend["product"]["prev_avg"])
    if trend["artist"]:  print("  아티스트 트렌드: 이번주", trend["artist"]["this_avg"],  "/ 전주", trend["artist"]["prev_avg"])

    # 뉴스: 상품명 검색 / 아티스트명 검색 각각
    news_product = get_news(kw_prd)
    news_artist  = get_news(kw_art)
    print("  뉴스: 상품 {}건 / 아티스트 {}건".format(len(news_product), len(news_artist)))

    # 카페: 상품명 / 아티스트명 각각
    cafe_product = get_cafe(kw_prd)
    cafe_artist  = get_cafe(kw_art)
    print("  카페글: 상품 {}건 / 아티스트 {}건".format(len(cafe_product), len(cafe_artist)))

    # Claude 분석
    print("  Claude 분석 중...")
    ai = analyze(product, trend, news_product, news_artist, cafe_product, cafe_artist)

    results.append({
        "product":       product,
        "trend":         trend,
        "news_product":  news_product,
        "news_artist":   news_artist,
        "cafe_product":  cafe_product,
        "cafe_artist":   cafe_artist,
        "analysis":      ai,
    })
    print("  완료")

print("\n✅ 전체 분석 완료")

In [ ]:
# ══════════════════════════════════════════════
# CELL 5  HTML 리포트 저장
# ══════════════════════════════════════════════

def arrow(pct):
    if pct > 5:  return '<span style="color:#16a34a;font-weight:700">▲{}%</span>'.format(abs(pct))
    if pct < -5: return '<span style="color:#dc2626;font-weight:700">▼{}%</span>'.format(abs(pct))
    return '<span style="color:#64748b;font-weight:700">―{}%</span>'.format(abs(pct))

def spark(daily, color="#6366f1"):
    if not daily: return ""
    vals = [d["ratio"] for d in daily]
    mn, mx = min(vals), max(vals); rng = mx-mn or 1
    W, H = 90, 26
    pts = ["{:.1f},{:.1f}".format(j/(max(len(vals)-1,1))*W, H-((v-mn)/rng*(H-4)+2)) for j,v in enumerate(vals)]
    return '<svg width="{}" height="{}" viewBox="0 0 {} {}"><polyline points="{}" fill="none" stroke="{}" stroke-width="1.5" stroke-linejoin="round"/></svg>'.format(W,H,W,H," ".join(pts),color)

def trend_row(t, label, color):
    if not t: return "<div style='font-size:11px;color:#94a3b8'>{}: 데이터 없음</div>".format(label)
    return """
    <div style='display:flex;align-items:center;justify-content:space-between;background:#f8faff;border:1px solid #e0e7ff;border-radius:6px;padding:8px 12px;margin-bottom:6px'>
      <div>
        <div style='font-size:10px;color:#64748b;margin-bottom:3px'>{}</div>
        <div style='font-size:12px'>이번주 <b>{}</b> vs 전주 <b>{}</b> &nbsp;{}</div>
      </div>
      <div>{}</div>
    </div>""".format(label, t["this_avg"], t["prev_avg"], arrow(t["change_pct"]), spark(t.get("daily",[]), color))

def news_list(items):
    if not items: return "<div style='font-size:11px;color:#94a3b8'>없음</div>"
    li = "".join(["<li style='padding:3px 0;border-bottom:1px solid #f8fafc;font-size:11px'>"
                  "<a href='{}' target='_blank' style='color:#3b4ff8;text-decoration:none'>{}</a>"
                  " <span style='color:#94a3b8;font-size:10px'>{}</span></li>".format(n["link"],n["title"],n["date"]) for n in items])
    return "<ul style='list-style:none;padding:0;margin:0'>{}</ul>".format(li)

def cafe_list(items):
    if not items: return "<div style='font-size:11px;color:#94a3b8'>없음</div>"
    li = "".join(["<li style='padding:3px 0;border-bottom:1px solid #f8fafc;font-size:11px'>"
                  "<a href='{}' target='_blank' style='color:#7c3aed;text-decoration:none'>{}</a>"
                  " <span style='color:#94a3b8;font-size:10px'>{} · {}</span></li>".format(c["link"],c["title"],c["cafe"],c["date"]) for c in items])
    return "<ul style='list-style:none;padding:0;margin:0'>{}</ul>".format(li)

cards = ""
for rank, r in enumerate(results, 1):
    p  = r["product"]
    tr = r["trend"]
    ai_html = re.sub(r"\*\*(.+?)\*\*", r"<strong style='color:#4f46e5'>\1</strong>", r["analysis"]).replace("\n","<br>")
    kw = tr.get("kw_product", p["product_name"])

    cards += """
    <div style='background:#fff;border-radius:14px;border:1px solid #e2e8f0;margin-bottom:24px;overflow:hidden'>

      <!-- 헤더 -->
      <div style='background:#f8fafc;padding:14px 20px;border-bottom:1px solid #e2e8f0;display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:10px'>
        <div>
          <span style='font-size:10px;font-weight:800;color:#94a3b8;background:#e2e8f0;border-radius:4px;padding:2px 6px'>#{rank}</span>
          <span style='font-size:15px;font-weight:700;color:#0f172a;margin-left:8px'>{pname}</span>
          <span style='font-size:12px;color:#6366f1;margin-left:6px;font-weight:600'>{aname}</span>
        </div>
        <div style='display:flex;gap:18px'>
          <div style='text-align:center'><div style='font-size:14px;font-weight:700'>{buyer:,}</div><div style='font-size:10px;color:#94a3b8'>구매자</div></div>
          <div style='text-align:center'><div style='font-size:14px;font-weight:700'>{order:,}</div><div style='font-size:10px;color:#94a3b8'>주문</div></div>
          <div style='text-align:center'><div style='font-size:14px;font-weight:700'>₩{gmv:,}만</div><div style='font-size:10px;color:#94a3b8'>거래액</div></div>
        </div>
      </div>

      <!-- 바디 -->
      <div style='display:flex'>

        <!-- 왼쪽: 트렌드 + 뉴스 + 카페 -->
        <div style='flex:1;padding:16px 20px;min-width:0'>

          <div style='font-size:10px;font-weight:700;letter-spacing:1px;color:#94a3b8;margin-bottom:8px'>📈 네이버 검색 트렌드</div>
          {trend_prd}
          {trend_art}

          <div style='display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:16px'>
            <div>
              <div style='font-size:10px;font-weight:700;letter-spacing:1px;color:#94a3b8;margin-bottom:6px'>📰 뉴스 — 상품 ({kw})</div>
              {news_prd}
              <div style='font-size:10px;font-weight:700;letter-spacing:1px;color:#94a3b8;margin:10px 0 6px'>📰 뉴스 — 아티스트 ({aname})</div>
              {news_art}
            </div>
            <div>
              <div style='font-size:10px;font-weight:700;letter-spacing:1px;color:#94a3b8;margin-bottom:6px'>☕ 카페 — 상품 ({kw})</div>
              {cafe_prd}
              <div style='font-size:10px;font-weight:700;letter-spacing:1px;color:#94a3b8;margin:10px 0 6px'>☕ 카페 — 아티스트 ({aname})</div>
              {cafe_art}
            </div>
          </div>
        </div>

        <!-- 오른쪽: AI 해석 -->
        <div style='flex:0 0 300px;padding:16px 20px;background:#fafbff;border-left:1px solid #f1f5f9'>
          <div style='font-size:10px;font-weight:700;letter-spacing:1px;color:#94a3b8;margin-bottom:8px'>🤖 AI 해석</div>
          <div style='font-size:12px;line-height:1.85;color:#334155;background:#fff;border:1px solid #e0e7ff;border-radius:8px;padding:12px 14px'>{ai}</div>
        </div>

      </div>
    </div>""".format(
        rank=rank, pname=p["product_name"], aname=p["artist_name"],
        buyer=p["buyer_cnt"], order=p["order_cnt"], gmv=p["gmv"]//10000,
        trend_prd=trend_row(tr.get("product"), "상품 검색량 ({})".format(kw),    "#6366f1"),
        trend_art=trend_row(tr.get("artist"),  "아티스트 검색량 ({})".format(p["artist_name"]), "#f59e0b"),
        kw=kw, aname2=p["artist_name"],
        news_prd=news_list(r["news_product"]),  news_art=news_list(r["news_artist"]),
        cafe_prd=cafe_list(r["cafe_product"]),  cafe_art=cafe_list(r["cafe_artist"]),
        ai=ai_html,
    )

html = """<!DOCTYPE html><html lang='ko'><head><meta charset='UTF-8'>
<title>주간 트렌드 리포트 {s}~{e}</title></head>
<body style='font-family:Apple SD Gothic Neo,Malgun Gothic,Arial,sans-serif;background:#f1f5f9;padding:32px 24px;font-size:13px;color:#1e293b;max-width:1200px;margin:0 auto'>
<div style='margin-bottom:24px'>
  <div style='font-size:20px;font-weight:700'>📊 주간 실적 트렌드 리포트</div>
  <div style='font-size:12px;color:#64748b;margin-top:4px'>{s} ~ {e} &nbsp;|&nbsp; TOP {n}개 상품 (구매자수 기준) &nbsp;|&nbsp; 뉴스·카페글: 상품명 + 아티스트 각각 조회</div>
</div>
{cards}
<div style='text-align:right;font-size:11px;color:#94a3b8;margin-top:8px'>Claude API + Naver DataLab / 뉴스 / 카페</div>
</body></html>""".format(s=START_DATE, e=END_DATE, n=len(results), cards=cards)

fname = "report_{}_{}.html".format(START_DATE, END_DATE)
with open(fname, "w", encoding="utf-8") as f:
    f.write(html)

print("✅ 리포트 저장:", fname)